# PII Engine - Benchmark en Dataiku

Objetivo: ejecutar el motor PII directamente sobre el dataset `data_test_final` y mostrar los resultados en un DataFrame, sin pasar por la webapp y sin las 7 categorias de clasificacion de informacion.

Requisitos:
- Notebook en el mismo proyecto DSS que `data_test_final`.
- `pii_engine` cargado en Project Libraries.
- Managed folders `model_binary_v1`, `model_entity_v1` y `model_content_v1`.
- Code env con `pandas`, `polars`, `scikit-learn`, `joblib`, `lightgbm`, `cloudpickle`, `PyYAML` y `dataiku`.

Sugerencia: empezar con `LIMIT=20000` y `SAMPLE_SIZE=1000`; despues subir para comparar calidad y tiempo.

In [ ]:
import gc
import sys
import time

import dataiku
import pandas as pd
import pii_engine
from IPython.display import display
from pii_engine.adapters.dataiku_adapter import DataikuAdapter

DATASET_NAME = "data_test_final"
BINARY_FOLDER = "model_binary_v1"
ENTITY_FOLDER = "model_entity_v1"
CONTENT_FOLDER = "model_content_v1"

LIMIT = 20_000
SAMPLE_SIZE = 1_000
SEED = 42
SAMPLING = "head"  # "head" o "random"

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 220)

print("Python:", sys.executable)
print("pii_engine:", pii_engine.__file__)
print(
    f"Dataset: {DATASET_NAME} | limit={LIMIT} | "
    f"sample_size={SAMPLE_SIZE} | sampling={SAMPLING}"
)

In [ ]:
adapter = DataikuAdapter(
    binary_folder=BINARY_FOLDER,
    entity_folder=ENTITY_FOLDER,
    content_folder=CONTENT_FOLDER,
    max_read_rows=LIMIT,
    strict_models=True,
)

health = adapter.health()
health

In [ ]:
ds = dataiku.Dataset(DATASET_NAME)

t0 = time.perf_counter()
df = ds.get_dataframe(
    sampling=SAMPLING,
    limit=LIMIT,
    infer_with_pandas=False,
)
read_s = time.perf_counter() - t0

print(
    f"Filas leidas: {len(df):,} | Columnas: {df.shape[1]} | "
    f"Lectura: {read_s:.2f}s"
)
print(f"Memoria aprox: {df.memory_usage(deep=True).sum() / 1e6:.1f} MB")
df.head()

In [ ]:
t0 = time.perf_counter()
results = adapter.analyze_dataframe(
    df,
    table_name=DATASET_NAME,
    sample_size=SAMPLE_SIZE,
    seed=SEED,
)
analyze_s = time.perf_counter() - t0

res_df = pd.DataFrame([r.to_dict() for r in results])
print(f"Analisis: {analyze_s:.2f}s | Columnas analizadas: {len(res_df)}")
res_df

In [ ]:
cols = [
    "column",
    "pii",
    "prob_pii",
    "entity",
    "prob_entity",
    "evidence",
    "match_rate",
    "sample_size",
    "needs_review",
    "classification",
]
display(res_df[[c for c in cols if c in res_df.columns]])

In [ ]:
summary = {
    "filas_leidas": len(df),
    "columnas": df.shape[1],
    "pii": int(res_df["pii"].sum()),
    "revision": int(res_df["needs_review"].sum()),
    "sin_contenido": int((res_df["sample_size"] == 0).sum()),
    "lectura_s": round(read_s, 2),
    "analisis_s": round(analyze_s, 2),
    "total_s": round(read_s + analyze_s, 2),
}
pd.DataFrame([summary])

## Comparacion opcional de configuraciones

Esta celda puede tardar varios minutos. Ejecutala solo si quieres comparar limites, tamanos de muestra y metodos de lectura.

In [ ]:
def run_case(limit, sample_size, sampling):
    gc.collect()

    t0 = time.perf_counter()
    frame = ds.get_dataframe(
        sampling=sampling,
        limit=limit,
        infer_with_pandas=False,
    )
    t1 = time.perf_counter()

    case_results = adapter.analyze_dataframe(
        frame,
        table_name=DATASET_NAME,
        sample_size=sample_size,
        seed=SEED,
    )
    t2 = time.perf_counter()

    case_df = pd.DataFrame([r.to_dict() for r in case_results])
    return {
        "sampling": sampling,
        "limit": limit,
        "sample_size": sample_size,
        "rows_read": len(frame),
        "cols": frame.shape[1],
        "read_s": round(t1 - t0, 2),
        "analyze_s": round(t2 - t1, 2),
        "total_s": round(t2 - t0, 2),
        "pii_cols": int(case_df["pii"].sum()),
        "review_cols": int(case_df["needs_review"].sum()),
    }

cases = [
    (5_000, 1_000, "head"),
    (10_000, 1_000, "head"),
    (20_000, 1_000, "head"),
    (50_000, 1_000, "head"),
    (20_000, 2_000, "head"),
    (50_000, 2_000, "head"),
    (50_000, 1_000, "random"),
]

bench = pd.DataFrame([run_case(*case) for case in cases])
bench.sort_values("total_s")

## Como interpretar

- Si `read_s` es alto y `analyze_s` es bajo, el cuello de botella es la lectura del dataset.
- Si `analyze_s` es alto, conviene bajar `SAMPLE_SIZE` o revisar la cantidad de columnas.
- Si `pii_cols` no cambia entre 20k y 50k filas, puedes usar un limite menor para pruebas rapidas.
- `sampling="random"` puede ser mas lento que `head` porque Dataiku recorre mas filas para muestrear.
- Este notebook usa el motor directamente: veras `classification` del motor (1/2), no las 7 categorias de informacion de la webapp.